# Session 3 — Linear Regression Fundamentals, Then Applied to Real Data
### Day 1 | FDP: Machine Learning for Materials and Metallurgical Engineering

This session has two parts:
- **Part 0** — a review of the linear regression equation and loss, using the classic car-weight/fuel-efficiency example
- **Part 1** — applying the same ideas to a real, 198-point metallurgical dataset (Copper, Hall-Petch)

## Setup — Import Libraries
We need these throughout both Part 0 and Part 1, so let's load them once, right at the start.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


## Part 0 — Linear Regression Fundamentals (Review)

Linear regression finds the relationship between one or more features and a label. The general equation is:

$$y' = b + w_1 x_1$$

where:
- $y'$ — the predicted label (the output)
- $b$ — the **bias** (equivalent to the y-intercept)
- $w_1$ — the **weight** of the feature (equivalent to the slope)
- $x_1$ — the feature (the input)

**Note on notation:** in the Hall-Petch context (Part 1), we called these $\sigma_0$ and $k$ instead of $b$ and $w_1$ — same underlying math, different names because the equation comes from metallurgy rather than generic statistics.

### The Classic Example: Car Weight vs. Fuel Efficiency

Suppose we want to predict a car's fuel efficiency (miles per gallon) from its weight.

In [ ]:
weight_1000lbs = np.array([3.5, 3.69, 3.44, 3.43, 4.34, 4.42, 2.37])
mpg = np.array([18, 15, 18, 16, 15, 14, 24])

plt.figure(figsize=(5.5, 4))
plt.scatter(weight_1000lbs, mpg, color="darkred")
plt.xlabel("Weight (1000s of lbs)")
plt.ylabel("Fuel efficiency (mpg)")
plt.title("Car weight vs. fuel efficiency")
plt.tight_layout()
plt.show()


A fitted model for this data turns out to be $y' = 30 + (-3.6)(x_1)$ — i.e., $b = 30$, $w_1 = -3.6$. We can use this model to predict fuel efficiency for a car not in our dataset.

In [ ]:
b, w1 = 30, -3.6

def predict_mpg(weight_1000lbs):
    return b + w1 * weight_1000lbs

# Predict fuel efficiency for a 4000 lb car
predicted = predict_mpg(4.0)
print(f"Predicted fuel efficiency for a 4000 lb car: {predicted:.1f} mpg")


---
## Quick Check 0a

What parts of the linear regression equation actually get updated during training?

**(i)** The predictions
**(ii)** The bias and weights ($b$ and $w_1$)
**(iii)** The feature values

*Think about it, then check the answer below.*

**Answer: (ii)** — training searches for the bias and weight values that best fit the data. The feature values are the input data itself (fixed, not something training changes), and predictions are just the *output* of applying the current $b$ and $w_1$ to a feature value.

### Measuring Error: Types of Loss

Once we have a candidate line, we need to measure how wrong it is. Several standard ways to do this:

| Loss type | Definition | Equation |
|---|---|---|
| L1 Loss | Sum of absolute errors | $\sum \lvert \text{actual} - \text{predicted} \rvert$ |
| MAE (Mean Absolute Error) | Average of L1 losses | $\frac{1}{N}\sum \lvert \text{actual} - \text{predicted} \rvert$ |
| L2 Loss | Sum of squared errors | $\sum (\text{actual} - \text{predicted})^2$ |
| MSE (Mean Squared Error) | Average of L2 losses | $\frac{1}{N}\sum (\text{actual} - \text{predicted})^2$ |

We'll use **MSE** throughout this course — it's the most common default for regression problems.

### Choosing a Loss: Why Squaring Matters (Outliers)

Squaring the error (L2/MSE) penalizes large errors much more heavily than small ones. This matters a lot when a dataset has outliers. Let's see this directly:

In [ ]:
# Two small sets of errors (actual - predicted) for two hypothetical models
errors_without_outlier = np.array([1, -2, 1.5, -1, 2])
errors_with_outlier   = np.array([1, -2, 1.5, -1, 15])   # one large outlier error

mse_without = np.mean(errors_without_outlier ** 2)
mse_with    = np.mean(errors_with_outlier ** 2)

mae_without = np.mean(np.abs(errors_without_outlier))
mae_with    = np.mean(np.abs(errors_with_outlier))

print(f"Without outlier:  MSE = {mse_without:.2f}   MAE = {mae_without:.2f}")
print(f"With outlier:     MSE = {mse_with:.2f}   MAE = {mae_with:.2f}")


---
## Quick Check 0b

Based on the numbers above, which loss (MSE or MAE) is more sensitive to a single large outlier?

**(i)** MSE — squaring makes the one large error dominate the total
**(ii)** MAE — it grows fastest with a single outlier
**(iii)** Both are affected equally

*Think about it, then check the answer below.*

**Answer: (i)** — MSE roughly quadruples while MAE only grows modestly, for the same single outlier. This is worth knowing for real datasets like the one in Part 1: a handful of unusual literature measurements can disproportionately pull an MSE-based fit toward them.

---
## Part 1 — Applying This to Real Data: Hall-Petch Across a Literature Dataset (Copper)

**About this dataset:** This part uses real data compiled by Cordero, Knight & Schuh, *"Six decades of the Hall-Petch effect – a survey of grain-size strengthening studies on pure metals,"* International Materials Reviews, 2016. It aggregates **1,445 grain-size/strength measurements across 20 pure metals**, drawn from dozens of independent published studies spanning six decades – real literature data, not a clean textbook table.

**Today's focus: Copper (Cu)** – an FCC metal, with 198 measurements in this dataset, spanning grain sizes from 320 µm down to just 0.004 µm (4 nanometres – genuinely nanocrystalline).

**What's different from Day 1 morning's Hall-Petch notebook:** that used a clean, 7-point table. Today's dataset is *real* – it mixes several different test methods and comes from many independent research groups. Part of this session is learning how to find and prepare the subset of data you actually need from something much larger and messier.

## Step 1 — Load the full dataset
Everything loads directly from the course GitHub repo – no file upload needed.

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/vijindal/fdp-ml/main/day1/hallpetch_multimetal_data.csv"
df = pd.read_csv(DATA_URL)

print("Full dataset shape:", df.shape)
df.head()


## Step 2 — Fetching the Data You Need From a Big Dataset

This dataset has 1,445 rows across 20 metals and 5 test methods. In real research, you almost never work with *all* of it at once – you need to know how to reliably pull out just the subset relevant to your question. Below are several common ways to do that in pandas, each useful in different situations.

### 2a — Get an overview first: how many measurements per metal?

In [ ]:
df["element"].value_counts()


### 2b — Basic filtering: select rows where a column matches a value

In [ ]:
cu = df[df["element"] == "Cu"].copy()
print("Copper measurements:", cu.shape[0])
cu.head()


### 2c — Filtering on more than one condition at once

Use `&` (and) or `|` (or) between conditions, with each condition wrapped in parentheses.

In [ ]:
cu_tension_only = df[(df["element"] == "Cu") & (df["test_method"] == "TT")].copy()
print("Copper, tension test only:", cu_tension_only.shape[0], "rows")
cu_tension_only.head()


### 2d — The same filter, written with `.query()`
Some people find this syntax more readable for multiple conditions.

In [ ]:
cu_tension_query = df.query("element == 'Cu' and test_method == 'TT'")
print("Same result?", cu_tension_query.shape[0] == cu_tension_only.shape[0])


### 2e — Summary statistics on a subset

In [ ]:
cu["grain_size_um"].describe()


### 2f — Sorting to find extremes
Which Cu measurement has the smallest grain size in this dataset?

In [ ]:
cu.sort_values("grain_size_um").head(3)


---
## Quick Check 1

Which line of code would give you **only the Copper rows measured using Vickers hardness (VH)**?

**(i)** `df[df["test_method"] == "VH"]`
**(ii)** `df[(df["element"] == "Cu") & (df["test_method"] == "VH")]`
**(iii)** `df[(df["element"] == "Cu") | (df["test_method"] == "VH")]`

*Think about it, then check the answer below.*

**Answer: (ii)** — option (i) gets Vickers-hardness rows for *every* metal, not just Cu. Option (iii) uses `|` (or), which would return Cu rows *and* all VH rows from every other metal too – far too broad. Only (ii), using `&` (and) with both conditions, correctly narrows to Cu-and-VH-only.

## Step 3 — Linearize: compute grain size to the power -1/2
Same transform as this morning – now applied to a much larger, messier real dataset.

In [ ]:
cu["d_inv_sqrt"] = cu["grain_size_um"] ** -0.5
cu[["grain_size_um", "d_inv_sqrt", "flow_stress_MPa", "test_method", "reference"]].head()


## Step 4 — Visualize: raw vs. linearized, across all references and test methods

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

axes[0].scatter(cu["grain_size_um"], cu["flow_stress_MPa"], s=15, alpha=0.6, color="darkred")
axes[0].set_xlabel("Grain size (µm)")
axes[0].set_ylabel("Flow stress (MPa)")
axes[0].set_title("Raw relationship (nonlinear)")

axes[1].scatter(cu["d_inv_sqrt"], cu["flow_stress_MPa"], s=15, alpha=0.6, color="navy")
axes[1].set_xlabel("d$^{-1/2}$ (µm$^{-1/2}$)")
axes[1].set_ylabel("Flow stress (MPa)")
axes[1].set_title("Linearized relationship")

plt.tight_layout()
plt.show()


Notice this looks noisier than this morning's 7-point table – that's expected. This is 198 measurements from many independent labs, using different specimens, test methods, and strain rates. Real aggregated literature data always has more scatter than a single controlled experiment.

## Step 4b — Recall: Measuring Fit Quality With MSE

From Part 0: **MSE** (mean squared error) measures how wrong a candidate line is. Here, our candidate line predicts flow stress as $\hat{y} = \sigma_0 + k \cdot x$ (where $x = d^{-1/2}$):

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

Let's apply this directly to our 198 real Cu measurements.

In [ ]:
def mean_squared_error(sigma0_guess, k_guess, x, y_actual):
    y_pred = sigma0_guess + k_guess * x
    return np.mean((y_actual - y_pred) ** 2)


## Step 4c — Parameters Exploration: Try a Few Guesses by Hand

Before letting an algorithm search for the best (σ0, k), let's try a few guesses ourselves and see which one fits better – exactly what "fitting a model" means underneath.

In [ ]:
guesses = {
    "Guess A": (100, 50),
    "Guess B": (150, 70),
    "Guess C": (200, 40),
}

x = cu["d_inv_sqrt"].values
y = cu["flow_stress_MPa"].values

for name, (sigma0_guess, k_guess) in guesses.items():
    loss = mean_squared_error(sigma0_guess, k_guess, x, y)
    print(f"{name}: sigma_0={sigma0_guess}, k={k_guess}  ->  MSE = {loss:,.0f}")


---
## Quick Check 2

Based on the MSE values above, which guess fits the Cu data best – and what does "best" mean here?

**(i)** The guess with the highest MSE – more error means a more powerful model
**(ii)** The guess with the lowest MSE – the smallest average squared error between predictions and actual data
**(iii)** All three guesses are equally good since they're all somewhat close

*Think about it, then check the answer below.*

**Answer: (ii)** — lower MSE means the line's predictions are, on average, closer to the real measurements. "Fitting" a model always means searching for the parameters that make this error as small as possible – whether that search is done by hand (as we just did), by a closed-form formula, or by an iterative algorithm (which we'll build ourselves in Session 4).

## Step 5 — Let an Algorithm Find the Best Parameters

Trying guesses by hand doesn't scale – with 198 points, we want the parameters that minimize MSE *exactly*, not just "pretty good." `np.polyfit` solves exactly this: it searches for the (σ0, k) that gives the lowest possible MSE across all 198 points simultaneously.

In [ ]:
k_cu, sigma0_cu = np.polyfit(cu["d_inv_sqrt"], cu["flow_stress_MPa"], 1)

print(f"Copper (Cu) Hall-Petch fit:")
print(f"  sigma_0 = {sigma0_cu:.1f} MPa")
print(f"  k       = {k_cu:.1f} MPa·µm^0.5")


---
## Quick Check 3

Copper's fitted σ0 came out to a certain positive value, roughly consistent with copper's known single-crystal strength. What would it mean, physically, if a fit like this produced a **negative** σ0?

**(i)** Nothing unusual – negative values are common and meaningful here
**(ii)** It would suggest something is off – σ0 represents a physical strength floor and shouldn't be negative
**(iii)** It would mean the metal has no grain-boundary strengthening at all

*Think about it, then check the answer below.*

**Answer: (ii)** — σ0 represents the intrinsic lattice resistance to dislocation motion; a real metal's strength floor can't be negative. A negative fitted value would be a signal to check the data (mixed test methods without correction, outliers, or too few high-grain-size points) rather than a real physical result.

## Step 6 — Zooming Into the Nanocrystalline Regime

The dataset includes measurements down to 4 nanometres – far finer than the classical Hall-Petch studies of the 1950s. Does the *same* straight-line fit still describe these ultra-fine-grained points well, or does it break down?

In [ ]:
nano = cu[cu["grain_size_um"] < 1.0].copy()
coarse = cu[cu["grain_size_um"] >= 1.0].copy()

print(f"Nanocrystalline points (<1 µm): {len(nano)}")
print(f"Conventional/coarse points (>=1 µm): {len(coarse)}")

plt.figure(figsize=(6.5, 4.5))
plt.scatter(coarse["d_inv_sqrt"], coarse["flow_stress_MPa"], s=15, alpha=0.6, color="navy", label="Conventional grain size")
plt.scatter(nano["d_inv_sqrt"], nano["flow_stress_MPa"], s=20, alpha=0.8, color="crimson", label="Nanocrystalline (<1 µm)")

x_line = np.linspace(cu["d_inv_sqrt"].min(), cu["d_inv_sqrt"].max(), 100)
y_line = sigma0_cu + k_cu * x_line
plt.plot(x_line, y_line, color="black", linestyle="--", label="Full-dataset fit")

plt.xlabel("d$^{-1/2}$ (µm$^{-1/2}$)")
plt.ylabel("Flow stress (MPa)")
plt.title("Does the Hall-Petch line hold at the nanoscale?")
plt.legend()
plt.tight_layout()
plt.show()


---
## Quick Check 4

Looking at the plot above, how well does the single straight-line fit describe the nanocrystalline (red) points compared to the conventional (blue) points?

**(i)** Equally well – no visible difference
**(ii)** The nanocrystalline points scatter noticeably away from the line – the simple model is less reliable at this scale
**(iii)** The nanocrystalline points fit even better than the conventional ones

*This connects directly to what the source paper itself reports – discuss with a neighbor before checking the answer.*

**Answer: (ii)** — this is a real, documented finding: at very fine (nanocrystalline) grain sizes, measured strength increasingly deviates from the simple Hall-Petch line, sometimes falling below the extrapolated trend. This doesn't mean linear regression was done wrong – it means **the underlying physical model itself has a known limited range of validity**. Recognizing when a model's assumptions stop holding is just as important as fitting it correctly in the first place.

---
## (Optional) Stretch — Refit Using Only Tension-Test Data

Earlier we saw Cu's measurements mix several test methods (tension, Vickers hardness, compression, disk-bend). Does restricting to tension tests only – arguably the most directly comparable method – change the fitted σ0 and k noticeably?

In [ ]:
cu_tt = cu[cu["test_method"] == "TT"].copy()

k_tt, sigma0_tt = np.polyfit(cu_tt["d_inv_sqrt"], cu_tt["flow_stress_MPa"], 1)

print(f"All test methods:      sigma_0 = {sigma0_cu:.1f} MPa,  k = {k_cu:.1f}")
print(f"Tension tests only:    sigma_0 = {sigma0_tt:.1f} MPa,  k = {k_tt:.1f}")


## Wrap-Up

- Copper's Hall-Petch parameters (σ0, k) were fitted from **198 real literature measurements**, not a clean textbook table
- You practiced several ways to **fetch exactly the subset you need** from a large dataset – boolean filtering, multi-condition filtering, `.query()`, `.describe()`, and sorting
- The nanocrystalline points revealed a genuine **limit of the simple Hall-Petch model**

**Next session:** we bring in Iron (BCC) and Titanium (HCP) to compare Hall-Petch behavior across crystal structures – does copper's FCC structure behave differently from a BCC or HCP metal?